# Tarea 2: Generador de Nombres a Nivel de Caracteres con LSTM

**Curso:** Procesamiento de Lenguaje Natural — UCB  
**Entrega:** Semana 8  
**Valor:** 10% de la nota final  

---

## Objetivo

Construir un **modelo de lenguaje a nivel de caracteres** usando una red LSTM que sea capaz de:

- Aprender los patrones de nombres de personas en diferentes idiomas
- **Generar** nuevos nombres plausibles carácter por carácter
- Generar nombres **condicionados por idioma** (ej: generar un nombre que suene japonés, español, árabe, etc.)

## Contexto

Un modelo de lenguaje a nivel de caracteres predice el **siguiente carácter** dada una secuencia de caracteres previos:

$$P(c_t \mid c_1, c_2, \ldots, c_{t-1})$$

Esto es una aplicación directa de las arquitecturas **RNN/LSTM** y la **generación autoregresiva** que vimos en las Semanas 6 y 7.

## Dataset

Usaremos el dataset clásico de nombres por nacionalidad, que contiene miles de nombres humanos agrupados por idioma/país de origen. Lo descargaremos directamente desde los datos de los tutoriales de PyTorch.

## Rúbrica de Evaluación

| Sección | Puntos | Descripción |
|---------|:------:|-------------|
| **Parte 1:** Carga y Exploración de Datos | 10 | Carga correcta, EDA del dataset de nombres |
| **Parte 2:** Preprocesamiento a Nivel de Caracteres | 15 | Vocabulario, encoding, padding y DataLoader |
| **Parte 3:** Modelo LSTM | 25 | Implementación correcta del modelo CharLSTM |
| **Parte 4:** Entrenamiento | 15 | Loop de entrenamiento con teacher forcing |
| **Parte 5:** Generación y Análisis | 25 | Generación con temperatura, análisis cualitativo |
| **Parte 6:** Reflexión | 10 | Respuestas reflexivas y bien fundamentadas |
| **TOTAL** | **100** | |

---

## Configuración Inicial

Ejecuta esta celda para verificar las dependencias necesarias.

In [ ]:
# Instalación de dependencias (ejecutar solo una vez si es necesario)
# !uv pip install torch matplotlib pandas numpy seaborn

In [ ]:
# Imports necesarios
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import os
import random
import string
import unicodedata
import zipfile
import urllib.request
from pathlib import Path
from collections import Counter

# Configuración
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

# Reproducibilidad
SEED = 42
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')
print('✅ Todo listo!')

---

## Parte 1: Carga de Datos y Exploración (10 pts)

### 1.1 Descarga del dataset (3 pts)

Descarga el dataset de nombres del tutorial de PyTorch. Contiene archivos `.txt` con nombres agrupados por idioma (Arabic.txt, Chinese.txt, Spanish.txt, etc.).

**Requisitos:**
- Descargar y descomprimir el dataset
- Cargar todos los archivos en un diccionario `{idioma: [lista de nombres]}`
- Normalizar los caracteres Unicode a ASCII

In [ ]:
# --- Código proporcionado: descarga y carga del dataset ---

DATA_DIR = Path('data_nombres')
DATA_DIR.mkdir(exist_ok=True)

URL = 'https://download.pytorch.org/tutorial/data.zip'
ZIP_PATH = DATA_DIR / 'data.zip'

if not (DATA_DIR / 'data' / 'names').exists():
    print('Descargando dataset...')
    urllib.request.urlretrieve(URL, ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(DATA_DIR)
    print('✅ Descargado y descomprimido')
else:
    print('✅ Dataset ya existe')

# Función para convertir Unicode a ASCII
def unicode_to_ascii(s):
    """Convierte caracteres Unicode a ASCII equivalente."""
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
        and c in string.ascii_letters + " '.-"
    )

print(f"Ejemplo: 'Ślusàrski' → '{unicode_to_ascii('Ślusàrski')}'")
print(f"Ejemplo: 'José' → '{unicode_to_ascii('José')}'")

In [ ]:
# TODO: Carga los nombres de todos los archivos .txt en el directorio data/names/
# Cada archivo tiene un nombre por línea y el nombre del archivo es el idioma
# Ejemplo: Spanish.txt contiene nombres españoles, uno por línea
#
# Hint: usa Path.glob('*.txt') para encontrar los archivos
# Aplica unicode_to_ascii() a cada nombre

names_dir = DATA_DIR / 'data' / 'names'

# Diccionario: {idioma: [lista de nombres]}
nombres_por_idioma = {}  # TODO: Llena este diccionario

# Tu código aquí


# Verificación
print(f'Idiomas encontrados: {len(nombres_por_idioma)}')
for idioma, nombres in sorted(nombres_por_idioma.items()):
    print(f'  {idioma:15s}: {len(nombres):5d} nombres | Ej: {nombres[:3]}')

### 1.2 Análisis Exploratorio (7 pts)

Realiza un análisis exploratorio del dataset:

1. **Gráfico de barras** con la cantidad de nombres por idioma (ordenado de mayor a menor)
2. **Histograma** de la longitud de nombres (en caracteres) a través de todos los idiomas
3. **Distribución de caracteres** más frecuentes en todo el dataset

**Pregunta:** ¿Cuáles son los idiomas con más y menos nombres? ¿Cuál es la longitud típica de un nombre?

In [ ]:
# TODO 1.2.1: Gráfico de barras — cantidad de nombres por idioma


In [ ]:
# TODO 1.2.2: Histograma — distribución de longitud de nombres (en caracteres)


In [ ]:
# TODO 1.2.3: Distribución de los 30 caracteres más frecuentes
# Hint: usa Counter() sobre todos los caracteres de todos los nombres


**Tu respuesta sobre el EDA:**

*[Escribe tu respuesta aquí]*

---

## Parte 2: Preprocesamiento a Nivel de Caracteres (15 pts)

### 2.1 Vocabulario de caracteres (5 pts)

Construye un vocabulario de caracteres con los siguientes tokens especiales:

| Token | Índice | Significado |
|-------|:------:|-------------|
| `<PAD>` | 0 | Padding |
| `<SOS>` | 1 | Start of Sequence |
| `<EOS>` | 2 | End of Sequence |

Los caracteres regulares deben tener índices a partir de 3.

**Requisitos:**
- Crear mapeos `char_to_idx` y `idx_to_char`
- Incluir TODOS los caracteres que aparecen en el dataset
- Crear también un mapeo `lang_to_idx` para los idiomas

In [ ]:
# TODO: Construye el vocabulario de caracteres

# Tokens especiales
PAD_TOKEN = '<PAD>'
SOS_TOKEN = '<SOS>'
EOS_TOKEN = '<EOS>'
PAD_IDX = 0
SOS_IDX = 1
EOS_IDX = 2

# TODO: Recolecta todos los caracteres únicos del dataset
# Hint: itera sobre todos los nombres de todos los idiomas
all_chars = set()  # Tu código aquí


# TODO: Crea los mapeos char_to_idx e idx_to_char
# Los caracteres regulares empiezan en índice 3
char_to_idx = {PAD_TOKEN: PAD_IDX, SOS_TOKEN: SOS_IDX, EOS_TOKEN: EOS_IDX}
# Tu código aquí para agregar los caracteres


idx_to_char = {v: k for k, v in char_to_idx.items()}

# TODO: Crea el mapeo de idiomas
idiomas = sorted(nombres_por_idioma.keys())
lang_to_idx = {}  # Tu código aquí


VOCAB_SIZE = len(char_to_idx)
N_LANGS = len(lang_to_idx)

print(f'Tamaño del vocabulario: {VOCAB_SIZE} caracteres')
print(f'Número de idiomas: {N_LANGS}')
print(f'Primeros caracteres: {dict(list(char_to_idx.items())[:10])}')
print(f'Idiomas: {lang_to_idx}')

### 2.2 Codificación de nombres (5 pts)

Implementa una función que convierta un nombre (string) en una secuencia de índices, y otra que haga lo inverso.

**Formato de secuencia para entrenamiento:**
- **Input:** `<SOS> c1 c2 c3 ... cn` (lo que el modelo recibe)
- **Target:** `c1 c2 c3 ... cn <EOS>` (lo que el modelo debe predecir)

Esto significa que el modelo aprende a predecir el **siguiente carácter** dado los anteriores.

In [ ]:
# TODO: Implementa las funciones de codificación

def encode_nombre(nombre):
    """
    Convierte un nombre en dos tensores: input_seq y target_seq.
    
    Args:
        nombre: string con el nombre (ej: "María")
    
    Returns:
        input_seq: tensor [SOS, c1, c2, ..., cn]
        target_seq: tensor [c1, c2, ..., cn, EOS]
    """
    # TODO: Implementa
    pass


def decode_indices(indices):
    """
    Convierte una secuencia de índices de vuelta a un string.
    Ignora tokens especiales (PAD, SOS, EOS).
    
    Args:
        indices: lista o tensor de índices
    
    Returns:
        string con el nombre decodificado
    """
    # TODO: Implementa
    pass


# Prueba tus funciones
test_nombre = 'Carlos'
inp, tgt = encode_nombre(test_nombre)
print(f"Nombre: '{test_nombre}'")
print(f"Input:  {inp.tolist()} → {[idx_to_char[i.item()] for i in inp]}")
print(f"Target: {tgt.tolist()} → {[idx_to_char[i.item()] for i in tgt]}")
print(f"Decode: '{decode_indices(tgt)}'")

### 2.3 Dataset y DataLoader (5 pts)

Crea un `Dataset` de PyTorch que contenga todos los pares (nombre, idioma) y un `DataLoader` con padding.

**Requisitos:**
- El dataset debe retornar `(input_seq, target_seq, lang_idx)` para cada ejemplo
- Implementar una función `collate_fn` que haga padding de las secuencias al largo máximo del batch
- Separar en **train (80%)** y **validation (20%)**

In [ ]:
# TODO: Implementa el Dataset de PyTorch

class NamesDataset(Dataset):
    def __init__(self, data):
        """
        Args:
            data: lista de tuplas (nombre, idioma_string)
        """
        self.data = data
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        nombre, idioma = self.data[idx]
        # TODO: Retorna (input_seq, target_seq, lang_idx)
        # Hint: usa encode_nombre() y lang_to_idx
        pass


def collate_fn(batch):
    """
    Función de collate que hace padding de las secuencias.
    
    Args:
        batch: lista de tuplas (input_seq, target_seq, lang_idx)
    
    Returns:
        inputs_padded: (batch_size, max_len)
        targets_padded: (batch_size, max_len)
        langs: (batch_size,)
    """
    # TODO: Implementa el padding
    # Hint: usa torch.nn.utils.rnn.pad_sequence o hazlo manualmente con PAD_IDX
    pass


# --- Preparar datos ---

# Crear lista plana de (nombre, idioma)
all_data = []
for idioma, nombres in nombres_por_idioma.items():
    for nombre in nombres:
        all_data.append((nombre, idioma))

# Mezclar y separar train/val
random.shuffle(all_data)
split_idx = int(0.8 * len(all_data))
train_data = all_data[:split_idx]
val_data = all_data[split_idx:]

# Crear datasets y dataloaders
BATCH_SIZE = 128
train_dataset = NamesDataset(train_data)
val_dataset = NamesDataset(val_data)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f'Total ejemplos: {len(all_data)}')
print(f'Train: {len(train_data)} | Val: {len(val_data)}')

# Verificar un batch
sample_inputs, sample_targets, sample_langs = next(iter(train_loader))
print(f'\nBatch de ejemplo:')
print(f'  Inputs shape:  {sample_inputs.shape}')
print(f'  Targets shape: {sample_targets.shape}')
print(f'  Langs shape:   {sample_langs.shape}')

---

## Parte 3: Modelo LSTM para Generación de Caracteres (25 pts)

### 3.1 Arquitectura del modelo (20 pts)

Implementa un modelo `CharLSTM` con la siguiente arquitectura:

```
Entrada (índice de carácter + índice de idioma)
    ↓
Embedding de carácter (char_embed_dim)
    ↓
Embedding de idioma (lang_embed_dim) → Concatenar
    ↓
LSTM (hidden_dim, n_layers)
    ↓
Linear → logits (vocab_size)
```

**Requisitos:**
- Embedding separado para caracteres y para idiomas
- La información del idioma se **concatena** al embedding del carácter en cada paso temporal
- LSTM con `batch_first=True`
- Capa `Linear` de salida que produce logits sobre el vocabulario de caracteres
- El método `forward` recibe `(input_seqs, lang_idx)` y retorna `(logits, hidden_state)`

**Hiperparámetros sugeridos** (puedes modificarlos):
- `char_embed_dim = 32`
- `lang_embed_dim = 16`
- `hidden_dim = 128`
- `n_layers = 2`
- `dropout = 0.3`

In [ ]:
# TODO: Implementa el modelo CharLSTM

class CharLSTM(nn.Module):
    def __init__(self, vocab_size, n_langs, char_embed_dim=32, lang_embed_dim=16,
                 hidden_dim=128, n_layers=2, dropout=0.3):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        
        # TODO: Define las capas del modelo
        # 1. Embedding de caracteres: nn.Embedding(vocab_size, char_embed_dim)
        # 2. Embedding de idioma: nn.Embedding(n_langs, lang_embed_dim)
        # 3. LSTM: input_size = char_embed_dim + lang_embed_dim
        # 4. Linear: hidden_dim → vocab_size
        # 5. Dropout
        
        pass  # Reemplaza con tu implementación
    
    def forward(self, chars, lang_idx, hidden=None):
        """
        Args:
            chars: (batch, seq_len) — índices de caracteres
            lang_idx: (batch,) — índice del idioma
            hidden: tuple (h, c) o None
        
        Returns:
            logits: (batch, seq_len, vocab_size)
            hidden: tuple (h, c) actualizado
        """
        # TODO: Implementa el forward pass
        # 1. Obtener embeddings de caracteres: (batch, seq_len, char_embed_dim)
        # 2. Obtener embedding de idioma: (batch, lang_embed_dim)
        #    → Expandir a (batch, seq_len, lang_embed_dim) para concatenar
        #    Hint: lang_emb.unsqueeze(1).expand(-1, seq_len, -1)
        # 3. Concatenar: (batch, seq_len, char_embed_dim + lang_embed_dim)
        # 4. Pasar por LSTM
        # 5. Pasar por Linear para obtener logits
        
        pass  # Reemplaza con tu implementación


# Instanciar y verificar
model = CharLSTM(
    vocab_size=VOCAB_SIZE,
    n_langs=N_LANGS,
).to(device)

print(model)
print(f'\nParámetros totales: {sum(p.numel() for p in model.parameters()):,}')

### 3.2 Verificación del modelo (5 pts)

Verifica que el modelo funciona correctamente pasando un batch de ejemplo y comprobando las dimensiones de salida.

In [ ]:
# TODO: Pasa el batch de ejemplo por el modelo y verifica las dimensiones
# Hint: usa sample_inputs, sample_langs del DataLoader de arriba

# Mover a device
test_inputs = sample_inputs.to(device)
test_langs = sample_langs.to(device)

# Forward pass
# logits, hidden = model(...)

# Verificar dimensiones
# print(f'Input shape:  {test_inputs.shape}')    # (batch, seq_len)
# print(f'Output shape: {logits.shape}')          # (batch, seq_len, vocab_size)
# print(f'h shape: {hidden[0].shape}')            # (n_layers, batch, hidden_dim)
# print(f'c shape: {hidden[1].shape}')            # (n_layers, batch, hidden_dim)

---

## Parte 4: Entrenamiento (15 pts)

### 4.1 Loop de entrenamiento (10 pts)

Implementa el loop de entrenamiento con las siguientes especificaciones:

- **Loss:** `CrossEntropyLoss` con `ignore_index=PAD_IDX` (para ignorar el padding)
- **Optimizer:** Adam con `lr=0.003`
- **Gradient clipping:** `clip_grad_norm_` con max_norm=1.0
- **Épocas:** al menos 30 (ajustar según la convergencia)
- Registrar la loss de train y val en cada época

**Nota:** El modelo usa **teacher forcing implícito** porque recibe toda la secuencia input y predice toda la secuencia target de una vez (no genera carácter por carácter durante entrenamiento).

In [ ]:
# TODO: Implementa el loop de entrenamiento

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = optim.Adam(model.parameters(), lr=0.003)

N_EPOCHS = 40  # Ajustar si es necesario
train_losses = []
val_losses = []

for epoch in range(N_EPOCHS):
    # --- Train ---
    model.train()
    epoch_train_loss = 0
    n_train_batches = 0
    
    for inputs, targets, langs in train_loader:
        inputs = inputs.to(device)
        targets = targets.to(device)
        langs = langs.to(device)
        
        # TODO: Forward pass, calcular loss, backward, clip gradients, step
        # 1. logits, _ = model(inputs, langs)
        # 2. Reshape logits y targets para CrossEntropyLoss:
        #    logits: (batch * seq_len, vocab_size)
        #    targets: (batch * seq_len,)
        # 3. loss = criterion(logits_reshape, targets_reshape)
        # 4. optimizer.zero_grad(), loss.backward()
        # 5. torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        # 6. optimizer.step()
        
        pass  # Reemplaza
    
    avg_train_loss = epoch_train_loss / max(n_train_batches, 1)
    train_losses.append(avg_train_loss)
    
    # --- Validation ---
    model.eval()
    epoch_val_loss = 0
    n_val_batches = 0
    
    with torch.no_grad():
        for inputs, targets, langs in val_loader:
            inputs = inputs.to(device)
            targets = targets.to(device)
            langs = langs.to(device)
            
            # TODO: Forward pass y calcular loss (sin backward)
            pass  # Reemplaza
    
    avg_val_loss = epoch_val_loss / max(n_val_batches, 1)
    val_losses.append(avg_val_loss)
    
    if (epoch + 1) % 5 == 0:
        print(f'Época {epoch+1:3d}/{N_EPOCHS} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}')

print('\n✅ Entrenamiento completado')

### 4.2 Curvas de aprendizaje (5 pts)

Grafica las curvas de loss de entrenamiento y validación.

**Pregunta:** ¿El modelo está haciendo overfitting? ¿Cómo lo puedes saber a partir de las curvas?

In [ ]:
# TODO: Grafica las curvas de aprendizaje (train y val loss)


**Tu respuesta sobre overfitting:**

*[Escribe tu respuesta aquí]*

---

## Parte 5: Generación de Nombres y Análisis (25 pts)

### 5.1 Función de generación (10 pts)

Implementa una función `generar_nombre(modelo, idioma, temperatura, max_len)` que genere un nombre carácter por carácter usando el modelo entrenado.

**Algoritmo de generación autoregresiva:**

1. Comenzar con el token `<SOS>`
2. En cada paso:
   a. Pasar el carácter actual y el idioma por el modelo
   b. Obtener la distribución de probabilidad sobre el siguiente carácter
   c. Aplicar **temperatura** a los logits antes de softmax: $P(c) = \text{softmax}(z / \tau)$
   d. **Muestrear** el siguiente carácter de la distribución
3. Detener al generar `<EOS>` o alcanzar `max_len`

**Sobre la temperatura $\tau$:**
- $\tau = 1.0$: distribución original
- $\tau < 1.0$: distribución más "puntiaguda" → nombres más conservadores/predecibles
- $\tau > 1.0$: distribución más "plana" → nombres más creativos/diversos

In [ ]:
# TODO: Implementa la función de generación

def generar_nombre(modelo, idioma, temperatura=1.0, max_len=20):
    """
    Genera un nombre carácter por carácter usando el modelo.
    
    Args:
        modelo: el modelo CharLSTM entrenado
        idioma: string con el idioma (ej: 'Spanish')
        temperatura: float > 0 que controla la aleatoriedad
        max_len: longitud máxima del nombre generado
    
    Returns:
        string con el nombre generado
    """
    modelo.eval()
    with torch.no_grad():
        # TODO: Implementa la generación autoregresiva
        # 1. Preparar el input inicial: tensor con SOS_IDX
        # 2. Preparar el tensor de idioma
        # 3. Loop de generación:
        #    a. Forward pass (un carácter a la vez)
        #    b. Aplicar temperatura: logits = logits / temperatura
        #    c. Softmax → distribución de probabilidad
        #    d. Muestrear: torch.multinomial(probs, 1)
        #    e. Si el carácter es EOS → terminar
        #    f. Agregar el carácter a la secuencia
        # 4. Retornar el nombre como string
        
        pass  # Reemplaza con tu implementación


# Prueba rápida
print('Prueba de generación:')
for idioma in ['Spanish', 'Japanese', 'Arabic', 'English', 'Russian']:
    nombre = generar_nombre(model, idioma, temperatura=0.8)
    print(f'  {idioma:10s} → {nombre}')

### 5.2 Efecto de la temperatura (7 pts)

Genera **10 nombres** para **3 idiomas diferentes** usando **3 temperaturas** distintas: $\tau = 0.3, 0.8, 1.5$.

Muestra los resultados en una tabla y analiza cómo cambia la calidad y diversidad de los nombres.

In [ ]:
# TODO: Genera nombres con diferentes temperaturas

idiomas_test = ['Spanish', 'Japanese', 'Russian']  # Puedes cambiarlos
temperaturas = [0.3, 0.8, 1.5]

for idioma in idiomas_test:
    print(f'\n{"=" * 60}')
    print(f'Idioma: {idioma}')
    print(f'{"=" * 60}')
    for temp in temperaturas:
        nombres_gen = [generar_nombre(model, idioma, temperatura=temp) for _ in range(10)]
        print(f'  τ = {temp}: {nombres_gen}')

**Tu análisis del efecto de la temperatura:**

*[¿Qué diferencias observas entre temperaturas bajas y altas? ¿Cuál temperatura produce los nombres más "realistas"? ¿Cuál produce los más creativos?]*

### 5.3 Análisis cualitativo (8 pts)

Realiza un análisis más profundo de los nombres generados:

1. **Diversidad:** Para un idioma, genera 50 nombres con $\tau = 0.8$. ¿Cuántos nombres **únicos** se generaron? ¿Hay repeticiones?

2. **Distribución de longitud:** Compara la distribución de longitud de los nombres generados vs. los nombres reales para un idioma.

3. **Nombres reales vs. generados:** ¿Cuántos de los 50 nombres generados son nombres que **existen en el dataset**? (Queremos que el modelo genere nombres nuevos pero plausibles, no que simplemente memorice).

4. **Diferenciación por idioma:** Genera 10 nombres para 5 idiomas diferentes. ¿Se nota una diferencia en los patrones de los nombres según el idioma?

In [ ]:
# TODO 5.3.1: Diversidad — genera 50 nombres y cuenta los únicos


In [ ]:
# TODO 5.3.2: Distribución de longitud — histograma comparando reales vs generados


In [ ]:
# TODO 5.3.3: ¿Cuántos nombres generados existen en el dataset real?


In [ ]:
# TODO 5.3.4: Genera nombres para 5 idiomas y observa los patrones


**Tu análisis cualitativo:**

*[Escribe tus observaciones sobre diversidad, memorización vs. generalización, y diferenciación por idioma]*

---

## Parte 6: Preguntas de Reflexión (10 pts)

Responde las siguientes preguntas con al menos 3-4 oraciones cada una.

### Pregunta 1 (3 pts)

¿Por qué usamos un modelo a **nivel de caracteres** en lugar de a nivel de palabras para esta tarea? ¿Qué ventajas tiene para la generación de nombres? ¿En qué tipo de tareas sería mejor un modelo a nivel de palabras?

**Tu respuesta:**

*[Escribe aquí]*

### Pregunta 2 (3 pts)

Explica la diferencia entre **muestreo** (`torch.multinomial`) y **decodificación greedy** (`argmax`). ¿Por qué usamos muestreo para generar nombres en lugar de siempre tomar el carácter más probable? Relaciona tu respuesta con el concepto de temperatura.

**Tu respuesta:**

*[Escribe aquí]*

### Pregunta 3 (4 pts)

Compara este modelo (CharLSTM) con la arquitectura **Seq2Seq Codificador-Decodificador** que vimos en la Semana 7. ¿En qué se parecen y en qué se diferencian? ¿Se podría formular la generación de nombres condicionada por idioma como un problema Seq2Seq? ¿Cómo?

**Tu respuesta:**

*[Escribe aquí]*

---

## Bonus (hasta +10 pts extra)

Implementa **una** de las siguientes ideas:

1. **Generación con prefijo** (+5 pts): Implementa una función `completar_nombre(modelo, prefijo, idioma)` que reciba las primeras letras de un nombre y complete el resto. Por ejemplo: `completar_nombre(model, 'Car', 'Spanish')` → `'Carlos'`.

2. **Clasificación inversa** (+5 pts): Usa el modelo para estimar la probabilidad de un nombre dado un idioma: $P(\text{nombre} \mid \text{idioma})$. Luego, dado un nombre, determina cuál es el idioma más probable. Evalúa la accuracy de esta "clasificación" en el set de validación.

3. **Comparación LSTM vs. GRU** (+5 pts): Entrena una versión del modelo usando `nn.GRU` en lugar de `nn.LSTM`. Compara loss, calidad de generación y velocidad de entrenamiento.

In [ ]:
# BONUS (opcional)


---

## Instrucciones de Entrega

1. **Formato:** Entrega este notebook (`.ipynb`) con todas las celdas ejecutadas
2. **Código:** Todo el código debe ser ejecutable de principio a fin sin errores
3. **Respuestas:** Todas las preguntas de texto deben estar respondidas
4. **Gráficos:** Todos los gráficos deben estar visibles (no borrar outputs)
5. **Nombre del archivo:** `tarea02_APELLIDO_NOMBRE.ipynb`

---

*Procesamiento de Lenguaje Natural — UCB 2026*